# 02 — Feature Engineering

**Input:** raw `application_train.csv` and `application_test.csv`
**Output:** two processed feature matrices saved to `data/processed/`:
- `application_train_processed.parquet` — feature matrix + TARGET, NaNs preserved
- `application_test_processed.parquet` — same feature columns, no TARGET
- `application_train_imputed.parquet` — same as above but with median-imputed numerics (for the LR baseline, which requires non-null inputs)

**Why two versions of the train matrix?** LightGBM handles NaN natively — it learns the optimal split direction for missing values during training. Imputing for LightGBM throws away information. But logistic regression requires complete inputs. Rather than imputing on the fly inside `03_modeling.ipynb` (which would couple modelling and feature prep), we materialise both versions here. The cost is a small amount of extra disk.

## What this notebook applies, by EDA finding

| EDA finding | Action here |
|---|---|
| 365243 sentinel in `DAYS_EMPLOYED` | Clean + create `DAYS_EMPLOYED_ANOMALY` flag |
| `CODE_GENDER == 'XNA'` (4 rows in train) | Drop from train |
| EXT_SOURCE columns are top signals | Add `EXT_SOURCE_NA_COUNT` and `EXT_SOURCE_MEAN` |
| DTI-style intuition was weak | Still add `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM` — let the model decide |
| Age stratifies cleanly | `DAYS_BIRTH` already in the data, `DAYS_EMPLOYED_PERCENT` derived |
| Structurally missing building cols | Imputed with median for LR; NaN preserved for LightGBM |


## 1. Setup and load both tables

We load train and test together so every transformation is applied consistently. The test set has 121 columns (no `TARGET`); train has 122.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.data import load_application_train, load_application_test
from src.features import (
    clean_anomalies,
    add_derived_features,
    encode_categoricals,
    align_columns,
)

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be written to: {PROCESSED_DIR}")

Outputs will be written to: /Users/jeffrey/Documents/GitHub/credit-default-prediction-home-credit/data/processed


In [2]:
train_raw = load_application_train()
test_raw = load_application_test()

print(f"Train: {train_raw.shape[0]:,} rows x {train_raw.shape[1]} columns")
print(f"Test:  {test_raw.shape[0]:,} rows x {test_raw.shape[1]} columns")
assert "TARGET" in train_raw.columns
assert "TARGET" not in test_raw.columns

Train: 307,511 rows x 122 columns
Test:  48,744 rows x 121 columns


## 2. Drop `CODE_GENDER == 'XNA'` from train

EDA flagged 4 rows in train with `CODE_GENDER == 'XNA'`. With this volume (4 out of 307,511), any imputation introduces more assumption than it's worth. We drop them.

**Important:** we apply this to train only. The test set must always handle every category the model might see in production — if an XNA case appears at inference time, it should be encoded the same way as in training. Since we dropped XNA from train, it simply won't appear as a post-encoding column, and any test row with XNA will get zeros across all gender dummies.

In [3]:
xna_mask = train_raw["CODE_GENDER"] == "XNA"
print(f"Dropping {xna_mask.sum()} rows with CODE_GENDER == 'XNA'")
train_raw = train_raw[~xna_mask].reset_index(drop=True)
print(f"Train after drop: {train_raw.shape[0]:,} rows")

# Confirm test has no XNA rows (it might, in which case we'd need a different strategy)
test_xna = (test_raw["CODE_GENDER"] == "XNA").sum()
print(f"Test rows with XNA: {test_xna}")

Dropping 4 rows with CODE_GENDER == 'XNA'
Train after drop: 307,507 rows
Test rows with XNA: 0


## 3. Clean the `DAYS_EMPLOYED == 365243` sentinel

`src/features.py:clean_anomalies()` replaces the 365243 sentinel with NaN and adds a `DAYS_EMPLOYED_ANOMALY` binary flag. The flag carries the "is pensioner" signal that the EDA showed was associated with lower default risk.

In [4]:
train = clean_anomalies(train_raw)
test = clean_anomalies(test_raw)

# Verify the cleaning worked
print("Train DAYS_EMPLOYED summary after cleaning:")
print(train["DAYS_EMPLOYED"].describe().round(0))
print(f"\nAnomaly flag distribution in train: {train['DAYS_EMPLOYED_ANOMALY'].sum():,} rows flagged")
print(f"Anomaly flag distribution in test:  {test['DAYS_EMPLOYED_ANOMALY'].sum():,} rows flagged")

Train DAYS_EMPLOYED summary after cleaning:
count    252133.0
mean      -2384.0
std        2338.0
min      -17912.0
25%       -3175.0
50%       -1648.0
75%        -767.0
max           0.0
Name: DAYS_EMPLOYED, dtype: float64

Anomaly flag distribution in train: 55,374 rows flagged
Anomaly flag distribution in test:  9,274 rows flagged


## 4. Add derived features

Two groups of features:

**Ratio features** (from `src/features.py:add_derived_features`):
- `CREDIT_INCOME_RATIO` — credit amount / annual income (DTI-style)
- `ANNUITY_INCOME_RATIO` — annuity / annual income (payment burden)
- `CREDIT_TERM` — annuity / credit (implied loan term proxy)
- `DAYS_EMPLOYED_PERCENT` — days employed / age in days (employment stability proxy)

**EXT_SOURCE aggregations** (added here, not in src/features.py because they're specific to this dataset):
- `EXT_SOURCE_NA_COUNT` — count of missing EXT_SOURCE values (0, 1, 2, or 3). The EDA showed the *presence* of these scores is itself informative.
- `EXT_SOURCE_MEAN` — mean of available EXT_SOURCE values. Aggregating these into a single feature gives the LR baseline a usable composite, and gives LightGBM an additional split candidate.

The EDA noted that median DTI was similar between defaulters and non-defaulters. That doesn't mean DTI is useless — it means DTI alone doesn't separate the classes, but DTI in combination with EXT_SOURCE_2, age, and employment type may well do so. Tree models are good at finding this.

In [5]:
train = add_derived_features(train)
test = add_derived_features(test)

# EXT_SOURCE aggregations
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
for df_ in (train, test):
    df_["EXT_SOURCE_NA_COUNT"] = df_[ext_cols].isna().sum(axis=1)
    df_["EXT_SOURCE_MEAN"] = df_[ext_cols].mean(axis=1)

print("Train EXT_SOURCE_NA_COUNT distribution:")
print(train["EXT_SOURCE_NA_COUNT"].value_counts().sort_index())

# Sanity check on EXT_SOURCE_MEAN
print(f"\nEXT_SOURCE_MEAN — NaN count: {train['EXT_SOURCE_MEAN'].isna().sum()} "
      f"({train['EXT_SOURCE_MEAN'].isna().mean()*100:.2f}%)")

Train EXT_SOURCE_NA_COUNT distribution:
EXT_SOURCE_NA_COUNT
0    109587
1    161011
2     36737
3       172
Name: count, dtype: int64

EXT_SOURCE_MEAN — NaN count: 172 (0.06%)


**Sanity note:** `EXT_SOURCE_MEAN` is NaN only when all three source columns are NaN simultaneously. That should match the count of `EXT_SOURCE_NA_COUNT == 3` rows.

In [6]:
# Quick correlation check on the new features vs target
new_features = [
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM",
    "DAYS_EMPLOYED_PERCENT", "EXT_SOURCE_NA_COUNT", "EXT_SOURCE_MEAN",
    "DAYS_EMPLOYED_ANOMALY",
]
corr = train[new_features + ["TARGET"]].corr()["TARGET"].drop("TARGET")
print("Derived feature correlation with TARGET:")
print(corr.round(4).to_string())

Derived feature correlation with TARGET:
CREDIT_INCOME_RATIO     -0.0077
ANNUITY_INCOME_RATIO     0.0143
CREDIT_TERM              0.0127
DAYS_EMPLOYED_PERCENT   -0.0680
EXT_SOURCE_NA_COUNT      0.0281
EXT_SOURCE_MEAN         -0.2221
DAYS_EMPLOYED_ANOMALY   -0.0460


**Finding:** `EXT_SOURCE_MEAN` has the strongest correlation with TARGET of any feature we've seen so far (typically around -0.22, stronger than any of the individual EXT_SOURCE columns). This validates the aggregation as a useful feature.

`EXT_SOURCE_NA_COUNT` has positive correlation with default — applicants for whom we cannot retrieve external scores are higher risk. This is a known pattern: thin-file applicants default more often. Useful signal.

`DAYS_EMPLOYED_ANOMALY` has a moderate negative correlation — flagged-as-pensioner applicants are lower risk, as the EDA suggested.

## 5. One-hot encode categoricals

We use `pd.get_dummies` via the `src/features.py:encode_categoricals` helper, with `dummy_na=True` so missing values become their own column. For categoricals with high cardinality (`ORGANIZATION_TYPE` has ~58 unique values, `OCCUPATION_TYPE` has 18) this creates a lot of columns, but most have meaningful default-rate variation.

Alternative we're NOT using here: target encoding. It's powerful but introduces leakage risk if not done with proper out-of-fold encoding, and that's complexity we don't need for v1.

In [7]:
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(f"{len(cat_cols)} categorical columns to encode:")
for c in cat_cols:
    n_unique = train[c].nunique(dropna=False)
    print(f"  {c}: {n_unique} unique values")

16 categorical columns to encode:
  NAME_CONTRACT_TYPE: 2 unique values
  CODE_GENDER: 2 unique values
  FLAG_OWN_CAR: 2 unique values
  FLAG_OWN_REALTY: 2 unique values
  NAME_TYPE_SUITE: 8 unique values
  NAME_INCOME_TYPE: 8 unique values
  NAME_EDUCATION_TYPE: 5 unique values
  NAME_FAMILY_STATUS: 6 unique values
  NAME_HOUSING_TYPE: 6 unique values
  OCCUPATION_TYPE: 19 unique values
  WEEKDAY_APPR_PROCESS_START: 7 unique values
  ORGANIZATION_TYPE: 58 unique values
  FONDKAPREMONT_MODE: 5 unique values
  HOUSETYPE_MODE: 4 unique values
  WALLSMATERIAL_MODE: 8 unique values
  EMERGENCYSTATE_MODE: 3 unique values


In [9]:
# Encode train and test using the same set of categorical columns
train_encoded, _ = encode_categoricals(train, fit_columns=cat_cols)
test_encoded, _ = encode_categoricals(test, fit_columns=cat_cols)

print(f"Train after encoding:  {train_encoded.shape}")
print(f"Test after encoding:   {test_encoded.shape}")
print(f"\nColumn count delta (train vs test): "
      f"{train_encoded.shape[1] - test_encoded.shape[1]} columns")

Train after encoding:  (307507, 268)
Test after encoding:   (48744, 265)

Column count delta (train vs test): 3 columns


**Why the train/test column counts differ:** one-hot encoding creates a column per unique category. A category appearing only in train (or only in test) creates an asymmetric column set. The `align_columns()` helper in the next cell handles this — it intersects the column sets and drops any non-shared columns. This matches what happens at production inference time: the model only knows about features it saw during training.

## 6. Align train/test columns

In [10]:
train_aligned, test_aligned = align_columns(train_encoded, test_encoded)

print(f"Train after alignment: {train_aligned.shape}")
print(f"Test after alignment:  {test_aligned.shape}")
assert "TARGET" in train_aligned.columns
assert "TARGET" not in test_aligned.columns
assert train_aligned.shape[1] - 1 == test_aligned.shape[1], \
    "Train (minus TARGET) and test should have the same column count"
print("\nColumn counts match.")

Train after alignment: (307507, 266)
Test after alignment:  (48744, 265)

Column counts match.


## 7. Final sanity checks before saving

Before we write to disk, we verify the matrix is what we expect: no infinity values from division-by-zero in ratios, sensible memory footprint, and no unintentional column leakage.

In [11]:
# Check for infinities (from division by zero in our ratio features)
inf_check_cols = ["CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
                  "CREDIT_TERM", "DAYS_EMPLOYED_PERCENT"]
for col in inf_check_cols:
    n_inf = np.isinf(train_aligned[col]).sum()
    if n_inf > 0:
        print(f"WARNING: {col} has {n_inf} infinity values")
    else:
        print(f"OK: {col} has no infinities")

OK: CREDIT_INCOME_RATIO has no infinities
OK: ANNUITY_INCOME_RATIO has no infinities
OK: CREDIT_TERM has no infinities
OK: DAYS_EMPLOYED_PERCENT has no infinities


In [12]:
# Replace any infinities with NaN (a clean handling either way)
train_aligned = train_aligned.replace([np.inf, -np.inf], np.nan)
test_aligned = test_aligned.replace([np.inf, -np.inf], np.nan)

print("Train memory footprint:", round(train_aligned.memory_usage(deep=True).sum() / 1024**2, 1), "MB")
print("Test memory footprint: ", round(test_aligned.memory_usage(deep=True).sum() / 1024**2, 1), "MB")

Train memory footprint: 310.0 MB
Test memory footprint:  48.8 MB


In [13]:
# SK_ID_CURR is the applicant identifier - useful for joins but not a model feature
# We keep it in the matrix so we can recover identities at evaluation/scoring time
# but the modelling notebook will drop it before fitting.
print("ID column present in train:", "SK_ID_CURR" in train_aligned.columns)
print("ID column present in test: ", "SK_ID_CURR" in test_aligned.columns)

ID column present in train: True
ID column present in test:  True


## 8. Save the NaN-preserved matrices (for LightGBM)

LightGBM uses these directly — no imputation needed.

In [14]:
train_path = PROCESSED_DIR / "application_train_processed.parquet"
test_path = PROCESSED_DIR / "application_test_processed.parquet"

train_aligned.to_parquet(train_path, index=False)
test_aligned.to_parquet(test_path, index=False)

print(f"Saved: {train_path}  ({train_path.stat().st_size / 1024**2:.1f} MB)")
print(f"Saved: {test_path}   ({test_path.stat().st_size / 1024**2:.1f} MB)")

Saved: /Users/jeffrey/Documents/GitHub/credit-default-prediction-home-credit/data/processed/application_train_processed.parquet  (30.5 MB)
Saved: /Users/jeffrey/Documents/GitHub/credit-default-prediction-home-credit/data/processed/application_test_processed.parquet   (5.8 MB)


**Why parquet, not CSV:** parquet preserves dtypes, supports compression natively, and loads ~5-10x faster than CSV at this size. For a CV-heavy modelling notebook that reads the matrix repeatedly, this matters.

## 9. Build the imputed matrix for the LR baseline

Logistic regression cannot consume NaN. We impute with the **train median** (never the test median, which would be leakage). The same median is applied to test.

**Why median, not mean:** the financial columns are heavily right-skewed (we saw this in EDA — `AMT_INCOME_TOTAL` has max values 1000× the median). Median is the robust choice for right-skewed distributions.

We also need to scale features for LR — done in the modelling notebook, not here, because the scaler must be fitted on train folds within CV (otherwise we leak the test fold's variance into the train fold's scaling).

In [15]:
# Compute medians from train (excluding TARGET and the ID column)
feature_cols = [c for c in train_aligned.columns if c not in ("TARGET", "SK_ID_CURR")]
medians = train_aligned[feature_cols].median()

print(f"Computing medians over {len(feature_cols)} feature columns...")
print(f"Sample medians:\n{medians.head(5)}")

Computing medians over 264 feature columns...
Sample medians:
CNT_CHILDREN             0.0
AMT_INCOME_TOTAL    147150.0
AMT_CREDIT          513531.0
AMT_ANNUITY          24903.0
AMT_GOODS_PRICE     450000.0
dtype: float64


In [16]:
# Apply imputation
train_imputed = train_aligned.copy()
train_imputed[feature_cols] = train_imputed[feature_cols].fillna(medians)

# Same medians applied to test
test_imputed = test_aligned.copy()
test_imputed[feature_cols] = test_imputed[feature_cols].fillna(medians)

# Verify zero NaNs remain in feature columns
train_remaining_nans = train_imputed[feature_cols].isna().sum().sum()
test_remaining_nans = test_imputed[feature_cols].isna().sum().sum()
print(f"Remaining NaNs in train features: {train_remaining_nans}")
print(f"Remaining NaNs in test features:  {test_remaining_nans}")
assert train_remaining_nans == 0
assert test_remaining_nans == 0

Remaining NaNs in train features: 0
Remaining NaNs in test features:  0


In [17]:
# Save the imputed versions
train_imputed_path = PROCESSED_DIR / "application_train_imputed.parquet"
test_imputed_path = PROCESSED_DIR / "application_test_imputed.parquet"

train_imputed.to_parquet(train_imputed_path, index=False)
test_imputed.to_parquet(test_imputed_path, index=False)

print(f"Saved: {train_imputed_path}  ({train_imputed_path.stat().st_size / 1024**2:.1f} MB)")
print(f"Saved: {test_imputed_path}   ({test_imputed_path.stat().st_size / 1024**2:.1f} MB)")

Saved: /Users/jeffrey/Documents/GitHub/credit-default-prediction-home-credit/data/processed/application_train_imputed.parquet  (33.7 MB)
Saved: /Users/jeffrey/Documents/GitHub/credit-default-prediction-home-credit/data/processed/application_test_imputed.parquet   (6.3 MB)


## 10. Summary and what's next

### What was produced
- `data/processed/application_train_processed.parquet` — NaN-preserved, used by LightGBM
- `data/processed/application_test_processed.parquet` — corresponding test set
- `data/processed/application_train_imputed.parquet` — median-imputed, used by LR
- `data/processed/application_test_imputed.parquet` — corresponding test set

### Final feature matrix shape
~240 columns including the derived ratios, EXT_SOURCE aggregations, the anomaly flag, and one-hot encoded categoricals. `SK_ID_CURR` retained for downstream identification; `TARGET` retained in train only.

### Defensible choices recap
1. **Train-test transformations applied symmetrically** — same cleaning function, same encoding function, same column alignment. No information from test influences train.
2. **Median imputation on train statistics only** — no leakage from test medians into train.
3. **Two output matrices** — NaN-preserved for LightGBM (uses native NaN handling), median-imputed for LR (requires complete inputs). Avoids re-doing imputation inside CV folds.
4. **No feature selection** — let LightGBM's implicit selection do the work. Feature selection is a v2 topic.
5. **No bureau / previous_application features** — out of scope per README.

### What `03_modeling.ipynb` does next
- Load the appropriate parquet for each model (imputed for LR, NaN-preserved for LightGBM)
- Stratified 5-fold cross-validation
- Logistic regression baseline with `StandardScaler` fitted inside each fold
- LightGBM with class imbalance handling
- Optuna hyperparameter search on LightGBM (capped: 40 trials / 30 min budget)
- Save out-of-fold predictions and the final LightGBM model for `04_evaluation.ipynb` and `05_interpretation.ipynb`
